# logit-thermo 探索 Y の続き 3：文章の「相互作用の地図」と、結合の距離による弱まり方（マーデルング定数の下見、GPU、L4）

- **必要な secret：なし**（`HF_TOKEN` があれば取り込みが少し速くなるだけ）。**ランタイム：L4**（ハイメモリはいりません）。見込みは 60〜80 分（寝ているあいだに終わって、自動で切れます）。
- **探索**です。確かめには使いません。生成はせず、探索 Y と同じ BBC の記事の窓（38 本、元の順番）を 4 つのモデルに読ませるだけです。
- 本文のトークンを 1 つずつ（1 つおきの位置で）関係ないトークンに差し替えて、後ろのトークンの確率がどれだけ下がるかを測り、「結合の強さ」の地図を作ります。発想はオーナー（2026-10-28 22:31）。
- 読み方（R1・R2）は回す前に決めて、`scripts/a2_explore_lattice.py` の説明文に書いてあります。
- 中身は表示も保存もしません。保存先 `A2_explore/lattice_summary.json` と `A2_explore/lattice_{モデル}.npz`（数字だけ）。
- 計算のコード：`scripts/a2_2_collect.py`（`e578cfc480980be6…`）、`scripts/a2_explore_xy.py`（`5c5750e76358b86a…`）、`scripts/a2_explore_lattice.py`（`13bb55fe0e3d0a13…`）を使います（はじめの 2 つは、Drive にある探索 X・Y のノートブックから取り出して、ハッシュで確かめます。**`A2_explore_xy_L4.ipynb` を logit-thermo フォルダから動かさないでください**）。
- **通知**：🚀・モデルごとの ✅・🎉・⚠️。名札は「logit-thermo 探索 Y の続き 3（相互作用の地図・結合の弱まり方、4 モデル、L4）」。終わったら、またはエラーで止まったら、ランタイムを切ります。


In [ ]:
# セル1：通知、GPU、Drive
import os, time, urllib.request
NTFY_TOPIC = 'YOUR_NTFY_TOPIC'
NB_LABEL = 'logit-thermo 探索 Y の続き 3（相互作用の地図・結合の弱まり方、4 モデル、L4）'
def notify(msg, tags='pig'):
    try:
        req = urllib.request.Request('https://ntfy.sh/' + NTFY_TOPIC, data=(msg + '\n[' + NB_LABEL + ']').encode('utf-8'),
                                     headers={'Title': 'logit-thermo - explore lattice', 'Tags': tags})
        urllib.request.urlopen(req, timeout=10).read()
        print('通知を送りました：', msg)
    except Exception as e:
        print('通知は送れませんでした（実験には影響しません）：', type(e).__name__)
_NOTIFIED = {'error': False}
def _on_error(result):
    if result.error_in_exec is not None and not _NOTIFIED['error']:
        _NOTIFIED['error'] = True
        import traceback
        e = result.error_in_exec
        txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
        v = os.environ.get('HF_TOKEN')
        if v: txt = txt.replace(v, '***')
        txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
        notify('⚠️ 止まりました（エラー）：\n' + txt, tags='warning')
if not globals().get('_HOOK_ON'):
    get_ipython().events.register('post_run_cell', _on_error); _HOOK_ON = True
notify('🚀 スタートしました（これが届けば通知の設定は OK です）', tags='rocket')
os.environ.setdefault('PYTORCH_CUDA_ALLOC_CONF', 'expandable_segments:True')
import torch
if not torch.cuda.is_available() or torch.cuda.get_device_properties(0).total_memory < 14e9:
    name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'GPU なし'
    notify(f'⚠️ GPU が足りません（{name}）。L4 を選んでください。ランタイムを切ります', tags='warning')
    from google.colab import runtime; runtime.unassign()
    raise SystemExit
torch.backends.cuda.matmul.allow_tf32 = False; torch.backends.cudnn.allow_tf32 = False
torch.set_float32_matmul_precision('highest')
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
except Exception:
    pass

from google.colab import drive
drive.mount('/content/drive')


In [ ]:
# セル2：計算のコード
# a2_2_collect.py と a2_explore_xy.py は、Drive にある探索 X・Y のノートブック（A2_explore_xy_L4.ipynb）に埋め込んだものを
# 取り出して使う（ハッシュで中身が同じことを確かめる）。このノートブックに埋め込むのは a2_explore_lattice.py だけ。
import ast, glob, hashlib, json
_cands = sorted(glob.glob('/content/drive/MyDrive/logit-thermo' + '/**/A2_explore_xy_L4*.ipynb', recursive=True))
assert _cands, 'Drive の logit-thermo フォルダに A2_explore_xy_L4.ipynb が見つかりません'
_got = {}
for _p in _cands:
    for _c in json.load(open(_p))['cells']:
        _s = _c['source'] if isinstance(_c['source'], str) else ''.join(_c['source'])
        if 'SRC22 = ' not in _s:
            continue
        for _n in ast.parse(_s).body:
            if isinstance(_n, ast.Assign) and isinstance(_n.targets[0], ast.Name) and _n.targets[0].id in ('SRC22', 'SRCXY'):
                _got[_n.targets[0].id] = ast.literal_eval(_n.value)
    if len(_got) == 2:
        break
SRC22, SRCXY = _got['SRC22'], _got['SRCXY']
SRCT = '"""探索 Y の続き 3：文章の「相互作用の地図」と、結合の距離による弱まり方（マーデルング定数の下見）。2026-10-28 夜。確かめには使わない。\nGPU（L4）のノートブックで、scripts/a2_2_collect.py と scripts/a2_explore_xy.py を実行したのと同じ名前空間で実行する\n（load_docs・load_model・y_window・Y_PREFIX を使う）。生成はしない（人の文章を読むだけ）。発想はオーナー（2026-10-28 22:31）。\n\n材料：探索 Y と同じ BBC の窓（38 本）を元の順番のまま、「Article:\\\\n」のあとに置く。本文は先頭から最大 384 トークン。同じ 4 モデル。\n差し替える位置 i は 1 つおき（偶数番目）にする（計算を半分にするため。j はすべての位置）。\n相互作用の地図：本文の位置 i のトークンを、関係ないトークンに差し替えて読ませ直し、後ろの位置 j（j > i）の log p の下がり方\n  Δ(i, j) ＝ log p(x_j | 元の文) − log p(x_j | i を差し替えた文) を測る（温度 1。差を取るので、外す番号の処理はしない。float32）。\n  差し替え先：同じモデルのトークンで、ほかの記事の本文に出てくるトークン（その記事の本文に出てくるものは除く）から、出てくる回数に\n  比例して選ぶ（ふつうの言葉のまま、中身だけ関係なくする）。選び方を 2 通り（種）用意し、主には 2 つの平均を使う。\n距離の帯：トークンの距離 d ＝ j − i。帯（2、3、4〜5、6〜7、8〜11、12〜15、16〜23、24〜31、32〜47、48〜63、64〜95、96〜127）ごとに、\n  全記事の Δ をまとめた平均 D(帯) を出す。文の距離 k（j の文の番号 − i の文の番号、0〜12）ごとの平均 S(k) も出す。\n\n読み方（回す前に決める。探索の目安。95% 区間は記事を選び直すブートストラップ 1,000 回、種 20261101）：\n  R1 弱まり方の形：帯の D が正のところで、べき（log D ＝ a − α log d）と指数（log D ＝ b − d ÷ ξ）を当てはめ、log D の残差の二乗和を比べる。\n     4 モデルともべきのほうが小さい →「べき型（遠くまでゆっくり効く。イオン結晶型）」\n     4 モデルとも指数のほうが小さい →「指数型（すぐ消える。分子性結晶型）」\u3000それ以外 →「はっきりしない」\n     D が正でない帯が 3 つ以上あるモデルがあれば →「はっきりしない（遠くで符号が定まらない）」\n     α も記録する（1 以下なら、1 次元の並びでは足し算が収束しない形）。\n  R2 近くと遠くのどちらが多いか：近く N ＝ Σ_{d=1..16} D(d)、遠く F ＝ Σ_{d=17..256} D(d)（d ごとの平均を足す）。F ÷ N の 95% 区間が\n     4 モデルとも 1 より上 →「遠くの寄与が近くを上回る」、4 モデルとも 1 より下 →「近くが大半」、それ以外 →「はっきりしない」。\n  参考：文の距離ごとの S(k)。S(1) ÷ S(0)（隣の文と同じ文の比）、S(2) ÷ S(1)。差し替えの 2 通りで距離の形がどれだけ似ているか（相関）。\n保存するのは数だけ（記事ごとの距離の和と数、文の距離の和と数、地図そのものは float16 で）。中身は表示しない・保存しない。\n"""\nimport gc, json, math, time\nimport numpy as np\nimport torch\n\nL_MODELS = [(\'pythia-1b\', \'EleutherAI/pythia-1b\'), (\'qwen2.5-1.5b\', \'Qwen/Qwen2.5-1.5B\'),\n            (\'olmo2-1b\', \'allenai/OLMo-2-0425-1B\'), (\'smollm2-1.7b\', \'HuggingFaceTB/SmolLM2-1.7B\')]\nL_MAXTOK, L_BATCH, L_DRAWS, L_SEED, L_STRIDE = 384, 8, 2, 20261101, 2\nL_BOOT_SEED_OFFSET = 0\nL_BINS = [(2, 2), (3, 3), (4, 5), (6, 7), (8, 11), (12, 15), (16, 23), (24, 31), (32, 47), (48, 63), (64, 95), (96, 127)]\nL_KMAX = 12\nL_BOOT = 1000\n\n\n@torch.no_grad()\ndef token_logp(model, X, start):\n    """X：(B, L) のトークン。→ 位置 start..L−1 のトークンの log p、(B, L − start)、float32。"""\n    logits = model(X).logits[:, start - 1:X.shape[1] - 1].float()\n    lse = torch.logsumexp(logits, -1)\n    tgt = logits.gather(-1, X[:, start:, None])[:, :, 0]\n    out = (tgt - lse).cpu().numpy()\n    del logits, lse, tgt\n    return out\n\n\n@torch.no_grad()\ndef interaction_map(model, ids, start, rep, device, bs=L_BATCH, stride=L_STRIDE):\n    """ids：bos ＋ 書き出し ＋ 本文 のトークン列。本文は位置 start から n 個。rep：(n,) の差し替え先。→ Δ (n, n)、j ≤ i は nan。\n    差し替える位置 i は stride おき（計算を軽くするため。測らない行は nan のまま）。"""\n    X0 = torch.tensor([ids], device=device)\n    n = len(ids) - start\n    base = token_logp(model, X0, start)[0]\n    D = np.full((n, n), np.nan, np.float32)\n    pos = list(range(0, n, stride))\n    for a in range(0, len(pos), bs):\n        rows = pos[a:a + bs]\n        X = X0.repeat(len(rows), 1)\n        for r, i in enumerate(rows):\n            X[r, start + i] = int(rep[i])\n        lp = token_logp(model, X, start)\n        for r, i in enumerate(rows):\n            d = base - lp[r]\n            d[:i + 1] = np.nan\n            D[i] = d\n    return D\n\n\ndef rep_tokens(body_ids_all, j, n, rng):\n    own = set(body_ids_all[j])\n    pool = [t for k, b in enumerate(body_ids_all) if k != j for t in b if t not in own]\n    return np.array(rng.choice(pool, size=n, replace=True))\n\n\ndef sent_index(body_text, sents, offsets):\n    starts, pos = [], 0\n    for s in sents:\n        starts.append(pos); pos += len(s) + 1\n    starts = np.array(starts)\n    return np.array([int(np.searchsorted(starts, a, side=\'right\') - 1) for a, _ in offsets])\n\n\ndef doc_profiles(D, sidx):\n    """→ 距離 d（0..n−1）ごとの Δ の和と数、文の距離 k（0..L_KMAX）ごとの和と数。"""\n    n = D.shape[0]\n    dsum, dcnt = np.zeros(n), np.zeros(n)\n    for d in range(1, n):\n        v = np.diagonal(D, offset=d).astype(np.float64)\n        v = v[np.isfinite(v)]\n        dsum[d], dcnt[d] = v.sum(), len(v)\n    ksum, kcnt = np.zeros(L_KMAX + 1), np.zeros(L_KMAX + 1)\n    I, J = np.triu_indices(n, 1)\n    k = sidx[J] - sidx[I]\n    ok = (k <= L_KMAX) & np.isfinite(D[I, J])\n    np.add.at(ksum, k[ok], D[I[ok], J[ok]].astype(np.float64))\n    np.add.at(kcnt, k[ok], 1)\n    return dsum, dcnt, ksum, kcnt\n\n\ndef pooled(dsum, dcnt, docs=None):\n    """記事ごとの和と数（(記事, 長さ)）→ d ごとの平均（長さ）。"""\n    if docs is not None:\n        dsum, dcnt = dsum[docs], dcnt[docs]\n    s, c = dsum.sum(0), dcnt.sum(0)\n    with np.errstate(invalid=\'ignore\', divide=\'ignore\'):\n        return np.where(c > 0, s / c, np.nan)\n\n\ndef bin_profile(Dd):\n    out = []\n    for a, b in L_BINS:\n        v = Dd[a:b + 1]\n        out.append(float(np.nanmean(v)) if np.any(np.isfinite(v)) else float(\'nan\'))\n    return np.array(out)\n\n\ndef fit_forms(Db):\n    """帯の平均 → べきと指数の当てはめ。D が正の帯だけ使う。"""\n    centers = np.array([math.sqrt(a * b) for a, b in L_BINS])\n    ok = np.isfinite(Db) & (Db > 0)\n    res = {\'n_nonpos_bins\': int((~ok).sum())}\n    if ok.sum() < 4:\n        res.update(alpha=float(\'nan\'), xi=float(\'nan\'), rss_power=float(\'nan\'), rss_exp=float(\'nan\'))\n        return res\n    y = np.log(Db[ok]); x1 = np.log(centers[ok]); x2 = centers[ok]\n    A1 = np.vstack([np.ones_like(x1), x1]).T; A2 = np.vstack([np.ones_like(x2), x2]).T\n    c1, r1, *_ = np.linalg.lstsq(A1, y, rcond=None); c2, r2, *_ = np.linalg.lstsq(A2, y, rcond=None)\n    rss1 = float(((y - A1 @ c1) ** 2).sum()); rss2 = float(((y - A2 @ c2) ** 2).sum())\n    res.update(alpha=float(-c1[1]), xi=float(-1 / c2[1]) if c2[1] < 0 else float(\'inf\'), rss_power=rss1, rss_exp=rss2)\n    return res\n\n\ndef near_far(Dd):\n    N = float(np.nansum(Dd[1:17])); F = float(np.nansum(Dd[17:257]))\n    return N, F\n\n\ndef analyze_model_lattice(dsum, dcnt, ksum, kcnt):\n    nD = dsum.shape[0]\n    Dd = pooled(dsum, dcnt)\n    Db = bin_profile(Dd)\n    fit = fit_forms(Db)\n    N, F = near_far(Dd)\n    rng = np.random.default_rng(L_BOOT_SEED_OFFSET + L_SEED)\n    ratios, alphas, pw = [], [], []\n    for _ in range(L_BOOT):\n        idx = rng.integers(0, nD, nD)\n        Dd_b = pooled(dsum, dcnt, idx)\n        n_b, f_b = near_far(Dd_b)\n        ratios.append(f_b / n_b if n_b != 0 else float(\'nan\'))\n        fb = fit_forms(bin_profile(Dd_b))\n        alphas.append(fb[\'alpha\']); pw.append(fb[\'rss_power\'] < fb[\'rss_exp\'] if np.isfinite(fb[\'rss_power\']) else np.nan)\n    q = lambda v: [float(np.nanpercentile(v, 2.5)), float(np.nanpercentile(v, 97.5))]\n    S = np.where(kcnt.sum(0) > 0, ksum.sum(0) / np.maximum(kcnt.sum(0), 1), np.nan)\n    return dict(D_bins=Db.tolist(), D_d1=float(Dd[1]), fit=fit, alpha_ci=q(alphas), power_wins_frac=float(np.nanmean(pw)),\n                near=N, far=F, far_over_near=F / N if N != 0 else float(\'nan\'), far_over_near_ci=q(ratios),\n                S_k=S.tolist(), S1_over_S0=float(S[1] / S[0]), S2_over_S1=float(S[2] / S[1]))\n\n\ndef readings(models):\n    M = list(models)\n    fits = [models[m][\'fit\'] for m in M]\n    if any(f[\'n_nonpos_bins\'] >= 3 for f in fits):\n        r1 = \'はっきりしない（遠くで符号が定まらない）\'\n    elif all(f[\'rss_power\'] < f[\'rss_exp\'] for f in fits):\n        r1 = \'べき型（遠くまでゆっくり効く。イオン結晶型）\'\n    elif all(f[\'rss_exp\'] < f[\'rss_power\'] for f in fits):\n        r1 = \'指数型（すぐ消える。分子性結晶型）\'\n    else:\n        r1 = \'はっきりしない\'\n    cis = [models[m][\'far_over_near_ci\'] for m in M]\n    if all(lo > 1 for lo, _ in cis):\n        r2 = \'遠くの寄与が近くを上回る\'\n    elif all(hi < 1 for _, hi in cis):\n        r2 = \'近くが大半\'\n    else:\n        r2 = \'はっきりしない\'\n    return r1, r2\n\n\ndef run_lattice(docs, outdir, device, notify=print):\n    wins = [w for w in (y_window(d[\'text\']) for d in docs) if w is not None]\n    texts = [\' \'.join(s for p in w for s in p) for w in wins]\n    sents_all = [[s for p in w for s in p] for w in wins]\n    res = {\'n_docs\': len(wins), \'models\': {}, \'draw_corr\': {}}\n    for name, hf_id in L_MODELS:\n        t0 = time.time()\n        tok, model = load_model(hf_id, device)\n        pre = tok(Y_PREFIX, add_special_tokens=False).input_ids\n        bos = [tok.bos_token_id] if (tok.bos_token_id is not None and getattr(tok, \'add_bos_token\', False)) else []\n        encs = [tok(t, add_special_tokens=False, return_offsets_mapping=True) for t in texts]\n        bodies = [e[\'input_ids\'][:L_MAXTOK] for e in encs]\n        sidx = [sent_index(t, s, e[\'offset_mapping\'][:L_MAXTOK]) for t, s, e in zip(texts, sents_all, encs)]\n        n_max = max(len(b) for b in bodies)\n        dsum = np.zeros((L_DRAWS, len(wins), n_max)); dcnt = np.zeros_like(dsum)\n        ksum = np.zeros((L_DRAWS, len(wins), L_KMAX + 1)); kcnt = np.zeros_like(ksum)\n        maps = []\n        for j, b in enumerate(bodies):\n            Ds = []\n            for g in range(L_DRAWS):\n                rng = np.random.default_rng(L_SEED + 1000 * g + j)\n                rep = rep_tokens(bodies, j, len(b), rng)\n                D = interaction_map(model, bos + pre + b, len(bos) + len(pre), rep, device)\n                a1, a2, a3, a4 = doc_profiles(D, sidx[j])\n                dsum[g, j, :len(a1)], dcnt[g, j, :len(a2)], ksum[g, j], kcnt[g, j] = a1, a2, a3, a4\n                Ds.append(D)\n            maps.append(np.nanmean(np.stack(Ds), 0).astype(np.float16))\n        # 主：2 通りの平均（和と数を足し合わせる）\n        r = analyze_model_lattice(dsum.sum(0), dcnt.sum(0), ksum.sum(0), kcnt.sum(0))\n        p0, p1 = bin_profile(pooled(dsum[0], dcnt[0])), bin_profile(pooled(dsum[1], dcnt[1]))\n        res[\'draw_corr\'][name] = float(np.corrcoef(p0, p1)[0, 1])\n        r[\'seconds\'] = round(time.time() - t0, 1)\n        res[\'models\'][name] = r\n        np.savez_compressed(f\'{outdir}/lattice_{name}.npz\', dsum=dsum, dcnt=dcnt, ksum=ksum, kcnt=kcnt,\n                            ntok=np.array([len(b) for b in bodies]), **{f\'map{j}\': m for j, m in enumerate(maps)})\n        del model; gc.collect(); torch.cuda.empty_cache()\n        notify(f\'{name} おわり（{time.time() - t0:.0f} 秒）\')\n    res[\'R1\'], res[\'R2\'] = readings(res[\'models\'])\n    return res\n\n\n@torch.no_grad()\ndef self_test_lattice(device):\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n    hf_id = \'hf-internal-testing/tiny-random-GPT2LMHeadModel\'\n    tok = AutoTokenizer.from_pretrained(hf_id)\n    model = AutoModelForCausalLM.from_pretrained(hf_id, torch_dtype=torch.float32).to(device).eval()\n    # token_logp を素直な計算と比べる\n    ids = list(range(10, 40)); start = 5\n    lp = token_logp(model, torch.tensor([ids], device=device), start)[0]\n    z = model(torch.tensor([ids], device=device)).logits[0].double()\n    ls = torch.log_softmax(z, -1)\n    ref = np.array([float(ls[t - 1, ids[t]]) for t in range(start, len(ids))])\n    assert np.allclose(lp, ref, atol=1e-4), (lp[:3], ref[:3])\n    # 地図：差し替えた位置より前は変わらない（向きの確認）。そのため一時的に nan を入れない版を計算する\n    rep = np.array([3] * (len(ids) - start))\n    X0 = torch.tensor([ids], device=device); base = token_logp(model, X0, start)[0]\n    for i in [0, 7, 20]:\n        X = X0.clone(); X[0, start + i] = 3\n        d = base - token_logp(model, X, start)[0]\n        assert np.all(np.abs(d[:i]) < 1e-5), (\'前の位置が変わった\', i, d[:i])\n    D = interaction_map(model, ids, start, rep, device, bs=4, stride=1)\n    assert np.all(np.isnan(D[np.tril_indices(len(rep))])), \'対角とその下は nan のはず\'\n    X = X0.clone(); X[0, start + 7] = 3\n    assert abs(D[7, 12] - (base[12] - token_logp(model, X, start)[0][12])) < 1e-5\n    D2 = interaction_map(model, ids, start, rep, device, bs=4, stride=2)\n    assert np.all(np.isnan(D2[1])) and abs(D2[6, 12] - D[6, 12]) < 1e-5            # 1 つおきの行だけ測る\n    s2, c2, _, _ = doc_profiles(D2, np.zeros(len(rep), int))\n    assert np.all(np.isfinite(s2)) and c2[1] == len(range(0, len(rep) - 1, 2))\n    # 文の番号\n    sents = [\'Aa bb.\', \'Cc dd ee.\', \'Ff.\']\n    text = \' \'.join(sents)\n    offs = [(0, 2), (3, 6), (7, 9), (10, 12), (13, 16), (17, 20)]\n    assert sent_index(text, sents, offs).tolist() == [0, 0, 1, 1, 1, 2]\n    # 距離の和と文の距離の和\n    Dm = np.full((4, 4), np.nan, np.float32)\n    Dm[0, 1], Dm[0, 2], Dm[0, 3], Dm[1, 2], Dm[1, 3], Dm[2, 3] = 1, 2, 3, 4, 5, 6\n    s, c, ks, kc = doc_profiles(Dm, np.array([0, 0, 1, 1]))\n    assert s.tolist() == [0, 1 + 4 + 6, 2 + 5, 3] and c.tolist() == [0, 3, 2, 1]\n    assert ks[0] == 1 + 6 and kc[0] == 2 and ks[1] == 2 + 3 + 4 + 5 and kc[1] == 4\n    # 当てはめ：べきの形と指数の形を見分ける\n    d = np.arange(0, 300, dtype=float)\n    pw = np.where(d > 0, 2.0 * np.maximum(d, 1) ** -0.8, np.nan)\n    ex = np.where(d > 0, 2.0 * np.exp(-d / 6.0), np.nan)\n    fp, fe = fit_forms(bin_profile(pw)), fit_forms(bin_profile(ex))\n    assert fp[\'rss_power\'] < fp[\'rss_exp\'] and abs(fp[\'alpha\'] - 0.8) < 0.05, fp\n    assert fe[\'rss_exp\'] < fe[\'rss_power\'], fe\n    # 判定の言葉\n    mk = lambda f, ci: {\'fit\': f, \'far_over_near_ci\': ci}\n    m = {k: mk(fp, [1.2, 2.0]) for k in \'abcd\'}\n    assert readings(m) == (\'べき型（遠くまでゆっくり効く。イオン結晶型）\', \'遠くの寄与が近くを上回る\')\n    m = {k: mk(fe, [0.1, 0.5]) for k in \'abcd\'}\n    assert readings(m) == (\'指数型（すぐ消える。分子性結晶型）\', \'近くが大半\')\n    # 差し替え先は、その記事に出てこないトークン\n    bodies = [[1, 2, 3], [3, 4, 5], [6, 7]]\n    r = rep_tokens(bodies, 0, 50, np.random.default_rng(0))\n    assert not (set(r.tolist()) & {1, 2, 3}) and set(r.tolist()) <= {4, 5, 6, 7}\n    return \'OK\'\n'
assert hashlib.sha256(SRC22.encode()).hexdigest() == 'e578cfc480980be6b0d1e18377b6beb8b7555e0744f56b13f63cb304a9f73d53'
assert hashlib.sha256(SRCXY.encode()).hexdigest() == '5c5750e76358b86a1840e6894f9cd86c377e6be1f0b9d4104998b5144a1faaea'
assert hashlib.sha256(SRCT.encode()).hexdigest() == '13bb55fe0e3d0a137c7f7269fb27c84a6e6a1e0cfd57ddeeedcbe54de919aed3'
exec(SRC22)
exec(SRCXY)
exec(SRCT)
print('コードを置きました（', _p, 'から 2 つ取り出しました）')


In [ ]:
# セル3：自己テスト → 記事（A2-2 と同じか確かめる）→ 4 モデルで相互作用の地図 → 保存（結果は表示しない）
import json
try:
    print('自己テスト：', self_test('cuda'), self_test_xy('cuda'), self_test_lattice('cuda'))
    os.makedirs('/content/drive/MyDrive/logit-thermo/A2_explore', exist_ok=True)
    docs, meta = load_docs()
    old = json.load(open('/content/drive/MyDrive/logit-thermo' + '/A2_2/docs_meta.json'))
    assert [d['text_sha256'] for d in old['docs']] == [d['text_sha256'] for d in meta['docs']], 'BBC の記事が A2-2 と違います'
    r = run_lattice(docs, '/content/drive/MyDrive/logit-thermo/A2_explore', 'cuda', notify=lambda m: notify('✅ ' + m, tags='white_check_mark'))
    r['a2_explore_lattice_sha256'] = '13bb55fe0e3d0a137c7f7269fb27c84a6e6a1e0cfd57ddeeedcbe54de919aed3'
    json.dump(r, open('/content/drive/MyDrive/logit-thermo/A2_explore' + '/lattice_summary.json', 'w'), ensure_ascii=False, separators=(',', ':'))
    print('保存しました')
except Exception as e:
    import traceback
    _NOTIFIED['error'] = True
    txt = ''.join(traceback.format_exception(type(e), e, e.__traceback__))
    v = os.environ.get('HF_TOKEN')
    if v: txt = txt.replace(v, '***')
    txt = txt.encode('utf-8')[-3300:].decode('utf-8', 'ignore')
    notify('⚠️ 探索 Y の続き 3で止まりました。ランタイムを切ります：\n' + txt, tags='warning')
    if globals().get('AUTO_DISCONNECT', True):
        from google.colab import drive as _d, runtime as _r
        _d.flush_and_unmount(); time.sleep(3); _r.unassign()
    raise


In [ ]:
# 最後：終わりの通知と、ランタイムを切る
notify('🎉 終わりました：相互作用の地図を A2_explore/lattice_summary.json に保存しました', tags='tada')
if globals().get('AUTO_DISCONNECT', True):
    from google.colab import drive as _drive, runtime as _runtime
    _drive.flush_and_unmount(); time.sleep(3); _runtime.unassign()
